<a href="https://colab.research.google.com/github/bfreitas1098/AI-Tuberculosis-Lung-Disease-Detector/blob/main/CAP4630_AI_TB_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

cleaning and adjusting sizes of data sets

In [ ]:
import kagglehub
import cv2
import os
import glob
import numpy as np
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator # the data set
from sklearn.model_selection import train_test_split
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
base_folder = os.path.join(path, 'TB_Chest_Radiography_Database')
categories = ['Normal', 'Tuberculosis']

def load_initial_data(base_path, categories, img_size=224):
    X_list = []
    y_list = []
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
#load og images
    for idx, category in enumerate(categories):
        folder_path = os.path.join(base_path, category)
        image_files = glob.glob(os.path.join(folder_path, '*.png'))
        print(f"Loading {category} images...")
        for img_path in tqdm(image_files):
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
# clean
            img = clahe.apply(img)
            img = cv2.resize(img, (img_size, img_size))
            img = img.astype('float32') / 255.0
            X_list.append(img)
            y_list.append(idx)
# 0 none 1 tb
    return np.array(X_list), np.array(y_list)
X_all, y_all = load_initial_data(base_folder, categories)
#80% test 20% val/test
#train is testing data dont worry about temp
X_train, X_temp, y_train, y_temp = train_test_split(X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
# 10% Val, 10% Test from temp
#val is data for validation and test is testing data
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
# adds new tb images made from the test data (only in Training Set)
normal_train_count = np.sum(y_train == 0)
current_tb_train_count = np.sum(y_train == 1)
needed_tb = normal_train_count - current_tb_train_count
print(f"\nBalancing Training Set: Generating {needed_tb} synthetic Tuberculosis images...")
# setup augmentation (only for the tb)
datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='constant',
    cval=0
)
# isolate training tb images for augmentation
tb_train_indices = np.where(y_train == 1)[0]
tb_train_array = np.expand_dims(X_train[tb_train_indices], axis=-1)
#making the images
augmented_images = []
i = 0
if needed_tb > 0:
    for batch in datagen.flow(tb_train_array, batch_size=1):
        new_img = batch[0].reshape(224, 224)
        augmented_images.append(new_img)
        i += 1
        if i >= needed_tb:
            break
# update X_train and y_train with synthetic data
X_train = np.concatenate([X_train, np.array(augmented_images)])
y_train = np.concatenate([y_train, np.ones(len(augmented_images))])

print(f"\nFinal Dataset Stats:")
print(f"Training - Normal: {np.sum(y_train==0)} | Tuberculosis: {np.sum(y_train==1)}")
print(f"Validation: {len(X_val)} | Test: {len(X_test)}")

100%|██████████| 663M/663M [00:08<00:00, 78.9MB/s]

Extracting files...


Loading Normal images...


100%|██████████| 3500/3500 [00:45<00:00, 77.22it/s]


Loading Tuberculosis images...


100%|██████████| 700/700 [00:08<00:00, 80.31it/s]



Balancing Training Set: Generating 2240 synthetic Tuberculosis images...

Final Dataset Stats:
Training - Normal: 2800 | Tuberculosis: 2800
Validation: 420 | Test: 420
